# Module 5 — Inverse Kinematics
## Unit 5 — Numerical Inverse Kinematics in Practice
### Lesson 5.3 — Convergence, Step Size, and Failure Modes

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
A status-aware solver: converged / max_iter / diverged.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(t1, t2, L1, L2):
    return np.array([L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2)])

def jacobian_2link(t1, t2, L1, L2):
    s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12, -L2*s12],[L1*c1+L2*c12, L2*c12]])

def ik_2link_closed(x, y, L1, L2):
    c2=(x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
    if c2<-1-1e-9 or c2>1+1e-9: return []
    c2=max(-1.0,min(1.0,c2)); sols=[]
    for sign in (+1.0,-1.0):
        s2=sign*np.sqrt(max(0.0,1-c2*c2)); t2=np.arctan2(s2,c2)
        t1=np.arctan2(y,x)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
        sols.append((t1,t2))
        if abs(s2)<1e-6: break
    return sols

def reachable(x,y,L1,L2,tol=1e-9):
    r=np.hypot(x,y); return abs(L1-L2)-tol<=r<=L1+L2+tol

def ik_solve(target,theta0,L1,L2,alpha=1.0,tol=1e-5,max_iter=60):
    """Status-aware solver.
      'converged' : |e| < tol.
      'diverged'  : a *reachable* target whose error grows under too-large a step (oscillation).
      'max_iter'  : ran out of iterations without converging (e.g. an unreachable target, whose
                    error plateaus above tol)."""
    theta=np.array(theta0,float); target=np.array(target,float)
    reach = abs(L1-L2)-1e-9 <= np.hypot(*target) <= L1+L2+1e-9
    e0=np.linalg.norm(target-fk_two_link(theta[0],theta[1],L1,L2)); hist=[]; prev=e0
    for it in range(max_iter):
        e=np.linalg.norm(target-fk_two_link(theta[0],theta[1],L1,L2)); hist.append(e)
        if e<tol: return theta,"converged",it,hist
        if reach and it>3 and e>prev and e>e0: return theta,"diverged",it,hist
        prev=e
        J=jacobian_2link(theta[0],theta[1],L1,L2)
        theta=theta+alpha*np.linalg.pinv(J)@(target-fk_two_link(theta[0],theta[1],L1,L2))
    return theta,"max_iter",max_iter,hist

L1,L2=0.4,0.3
for (tgt,a,mx) in [([0.5,0.2],1.0,60),([0.5,0.2],0.3,200),([0.5,0.2],2.6,60),([0.9,0.0],1.0,60)]:
    th,st,it,_=ik_solve(tgt,np.radians([10,20]),L1,L2,alpha=a,max_iter=mx)
    print(f"target {tgt}, alpha {a}: {st} in {it} iters")


## Coding Exercise (§8)
Status matches each case: converged / slow / diverged / unreachable.


In [ ]:
# YOUR CODE HERE
th,st,it,_=ik_solve([0.5,0.2],np.radians([10,20]),L1,L2,alpha=1.0); assert st=="converged" and it<=6
# small alpha is slower: converges only with a generous cap
th,st2,it2,_=ik_solve([0.5,0.2],np.radians([10,20]),L1,L2,alpha=0.3,max_iter=200); assert st2=="converged" and it2>it
# unreachable target never converges (max_iter or diverged), and fails the reachability gate
th,st3,_,_=ik_solve([0.9,0.0],np.radians([10,20]),L1,L2,alpha=1.0); assert st3!="converged"
assert not reachable(0.9,0.0,L1,L2)
print("status logic OK")


## Check your work


In [ ]:
th,st,it,hist=ik_solve([0.5,0.2],np.radians([10,20]),L1,L2,alpha=1.0)
assert st=="converged" and hist[-1]<1e-5
print("All checks passed.")
